# 作业1：Data Types, Functions and Conditionals

**姓名：高琴　学号：20244080227**

按 HW01.pdf 的题目顺序给出实现、中文说明和运行结果。
所有算法使用 Python 标准库；请在独立的 `homework` conda 环境中从头运行全部单元。

## 身份与环境记录
机器指纹由当前执行电脑计算，换电脑运行时会随之变化。

In [1]:
import os, uuid, hashlib, platform, getpass

STUDENT_ID = 20244080227
STUDENT_NAME = "高琴"
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]     {STUDENT_ID}")
print(f"[姓名]     {STUDENT_NAME}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
assert CONDA_ENV != "base", "请切换到独立的 homework conda 环境"

[Conda 环境] homework
[学号]     20244080227
[姓名]     高琴
[机器指纹] 521b38693dc70995
[RUN_ID]   20244080227-521b38693dc70995


## 一、热身：定义简单函数

### 1. say_hi
不接收参数，调用时直接打印题目要求的问候语。

In [2]:
def say_hi():
    print("Hello, world!")


say_hi()

Hello, world!


### 2. goat_pad
将 `goat`、输入字符串和 `goat` 拼接后打印，不额外插入空格。

In [3]:
def goat_pad(s):
    print("goat" + s + "goat")


goat_pad("Python")
goat_pad("")

goatPythongoat
goatgoat


### 3. print_n
循环 n 次，每次执行一次 `print(s)`，因此每份字符串单独占一行。

In [4]:
def print_n(s, n):
    for _ in range(n):
        print(s)


print_n("Hello", 3)

Hello
Hello
Hello


## 二、欧几里得算法

### 1. 实现 gcd
以 `(48, 18)` 为例，不断把两个数换成“原来的第二个数”和“相除的余数”：

`(48, 18) → (18, 12) → (12, 6) → (6, 0)`。

第二个数变成 0 时，第一个数 6 就是最大公约数。
这样做的原因是：能同时整除两个数的整数，也能整除它们相除的余数，因此每次替换都保留最大公约数。
开始先取绝对值，让返回的最大公约数始终非负。

In [5]:
def gcd(m, n):
    m, n = abs(m), abs(n)
    while n != 0:
        m, n = n, m % n
    return m


print("gcd(48, 18) =", gcd(48, 18))

gcd(48, 18) = 6


### 2. 计算指定的三组输入

In [6]:
for m, n in ((2018, 2019), (1200, 300), (5040, 60)):
    print(f"gcd({m}, {n}) = {gcd(m, n)}")
assert gcd(2018, 2019) == 1
assert gcd(1200, 300) == 300
assert gcd(5040, 60) == 60

gcd(2018, 2019) = 1
gcd(1200, 300) = 300
gcd(5040, 60) = 60


### 3. 负数参数的行为
本实现先对输入取绝对值，因此 `gcd(-48, 18)`、`gcd(48, -18)` 和 `gcd(-48, -18)` 都返回 6。
这符合最大公约数通常取非负值的约定，改变输入的正负号不会改变它的正因数。

一个参数为 0 时返回另一个参数的绝对值。两个参数都是 0 时，数学上不存在最大的正公约数；
本实现采用与 Python `math.gcd` 一致的编程约定，返回 0。

In [7]:
for pair in ((-48, 18), (48, -18), (-48, -18), (0, -18), (-18, 0), (0, 0)):
    print(f"gcd{pair} = {gcd(*pair)}")

gcd(-48, 18) = 6
gcd(48, -18) = 6
gcd(-48, -18) = 6
gcd(0, -18) = 18
gcd(-18, 0) = 18
gcd(0, 0) = 0


## 三、近似欧拉数 e

### 1. 用极限公式近似：euler_limit
题目公式为 $e=\lim_{n\to\infty}(1+1/n)^n$。
函数直接计算正整数 $n$ 对应的 $(1+1/n)^n$，返回浮点数。
例如 $n=1$ 时得到 2，$n=10$ 时约为 2.594；在下面展示的范围内，增大 $n$ 会使结果更接近 e。
由于表达式有除法 `1/n`，这里对非正数给出明确错误。

In [8]:
import math


def euler_limit(n):
    if n <= 0:
        raise ValueError("n 必须为正整数")
    return (1 + 1 / n) ** n


true_e = math.exp(1)
print(f"参考值 e = {true_e:.15f}")
for n in (1, 10, 100, 1000, 10000):
    approximation = euler_limit(n)
    print(f"n={n:5d}, 近似值={approximation:.15f}, 绝对误差={abs(approximation - true_e):.8g}")

参考值 e = 2.718281828459045
n=    1, 近似值=2.000000000000000, 绝对误差=0.71828183
n=   10, 近似值=2.593742460100002, 绝对误差=0.12453937
n=  100, 近似值=2.704813829421528, 绝对误差=0.013467999
n= 1000, 近似值=2.716923932235594, 绝对误差=0.0013578962
n=10000, 近似值=2.718145926824926, 绝对误差=0.00013590163


### 2. 用级数前 n 项近似：euler_infinite_sum
这里的 n 是**项数**，因此实际相加的是 $k=0,1,\ldots,n-1$ 对应的 $1/k!$。
例如 n=4 时求 $1+1+1/2+1/6=8/3$，不包含 $1/4!$。

从 `term = 1.0`（$1/0!$）开始，每轮先相加，再除以 `k+1` 得到下一项。
这利用了 $1/(k+1)!=(1/k!)/(k+1)$，不用每次重新计算阶乘。
`n=0` 时循环不执行，返回浮点数 0.0；`n=1` 时只累加首项，返回 1.0。

In [9]:
def euler_infinite_sum(n):
    total = 0.0
    term = 1.0
    for k in range(n):
        total += term
        term /= k + 1
    return total


for n in (0, 1, 2, 4, 10):
    approximation = euler_infinite_sum(n)
    print(f"前 {n:2d} 项：{approximation:.15f}，类型：{type(approximation).__name__}")
assert euler_infinite_sum(0) == 0.0
assert euler_infinite_sum(1) == 1.0
assert math.isclose(euler_infinite_sum(4), 8 / 3)

前  0 项：0.000000000000000，类型：float
前  1 项：1.000000000000000，类型：float
前  2 项：2.000000000000000，类型：float
前  4 项：2.666666666666667，类型：float
前 10 项：2.718281525573192，类型：float


### 3. 按误差要求近似：euler_approx
例如要求误差小于 $10^{-6}$，就从首项 1.0 开始，每次增加下一项，并与 `math.exp(1)` 比较。

`当前和 → 检查绝对误差 → 若误差仍大则加下一项 → 再检查 → 返回满足精度的和`。

这里 `k` 是最后已加入的项的下标，`term` 是对应的 $1/k!$。
误差必须是有限的正数。如果误差要求小到浮点加法已无法让结果继续变化，函数明确报错，避免无限循环。
比较基准是题目允许使用的浮点参考值，不是无限精度的数学常数。

In [10]:
def euler_approx(epsilon):
    if not math.isfinite(epsilon) or epsilon <= 0:
        raise ValueError("epsilon 必须是有限的正数")
    target = math.exp(1)
    total = 1.0
    term = 1.0
    k = 0
    while abs(total - target) >= epsilon:
        k += 1
        term /= k
        next_total = total + term
        if next_total == total:
            raise ValueError("当前浮点精度无法满足指定误差")
        total = next_total
    return total


for epsilon in (0.1, 1e-3, 1e-6, 1e-12):
    approximation = euler_approx(epsilon)
    error = abs(approximation - true_e)
    print(f"epsilon={epsilon:g}, 近似值={approximation:.15f}, 绝对误差={error:.8g}")
    assert error < epsilon

epsilon=0.1, 近似值=2.666666666666667, 绝对误差=0.051615162
epsilon=0.001, 近似值=2.718055555555555, 绝对误差=0.0002262729
epsilon=1e-06, 近似值=2.718281525573192, 绝对误差=3.0288585e-07
epsilon=1e-12, 近似值=2.718281828458230, 绝对误差=8.149037e-13


### 4. 打印前 n 项和的表
从 k=1 到 n，每行打印 `euler_infinite_sum(k)`；不在函数内部添加额外表头。

In [11]:
def print_euler_sum_table(n):
    for k in range(1, n + 1):
        print(euler_infinite_sum(k))


print_euler_sum_table(10)

1.0
2.0
2.5
2.6666666666666665
2.708333333333333
2.7166666666666663
2.7180555555555554
2.7182539682539684
2.71827876984127
2.7182815255731922


## 四、测试整数属性

### 1. 判断是否为 2 的幂
例如 8 连续整除 2：`8 → 4 → 2 → 1`，所以是 2 的幂。
12 的过程是 `12 → 6 → 3`，3 不能再整除 2，因此不是。
每次用 `%` 判断是否整除，用 `//` 更新整数；不用平方根、对数或浮点判断。
1 也返回 True，因为它是 2 的 0 次幂。

In [12]:
def is_power_of_2(n):
    while n > 1 and n % 2 == 0:
        n //= 2
    return n == 1


for n in (1, 2, 3, 4, 8, 12, 16, 1024, 1025):
    print(f"{n} 是 2 的幂：{is_power_of_2(n)}")

1 是 2 的幂：True
2 是 2 的幂：True
3 是 2 的幂：False
4 是 2 的幂：True
8 是 2 的幂：True
12 是 2 的幂：False
16 是 2 的幂：True
1024 是 2 的幂：True
1025 是 2 的幂：False


### 2. 推广到任意正整数底数
参数顺序为 `is_power(b, n)`，判断 n 是否是 b 的非负整数次幂。
当 b>1 时重复整除 b，最后剩下 1 就返回 True。
例如 `is_power(3, 81)` 的过程为 `81 → 27 → 9 → 3 → 1`。

底数 1 单独处理：1 的任何非负整数次幂都等于 1，因此只有 n=1 才返回 True。
如果不单独处理，反复除以 1 不会使 n 变小，循环将无法结束。

In [13]:
def is_power(b, n):
    if b == 1:
        return n == 1
    while n > 1 and n % b == 0:
        n //= b
    return n == 1


for b, n in ((3, 81), (3, 45), (5, 125), (10, 1000), (2, 1), (1, 1), (1, 8)):
    print(f"is_power({b}, {n}) = {is_power(b, n)}")

is_power(3, 81) = True
is_power(3, 45) = False
is_power(5, 125) = True
is_power(10, 1000) = True
is_power(2, 1) = True
is_power(1, 1) = True
is_power(1, 8) = False


## 结果检查
使用标准库最大公约数作对照，检查正负数和零；用阶乘公式核对级数项数；
再用直接枚举幂的结果核对除法算法，并捕获打印内容检查函数输出格式。

In [14]:
import io
from contextlib import redirect_stdout


def captured_output(function, *args):
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        function(*args)
    return buffer.getvalue()


assert captured_output(say_hi) == "Hello, world!\n"
assert captured_output(goat_pad, "Python") == "goatPythongoat\n"
assert captured_output(goat_pad, "") == "goatgoat\n"
assert captured_output(print_n, "hi", 3) == "hi\nhi\nhi\n"
assert captured_output(print_n, "hi", 0) == ""
for m in range(-20, 21):
    for n in range(-20, 21):
        assert gcd(m, n) == math.gcd(m, n)
assert euler_limit(1) == 2.0
for n in range(21):
    expected = sum(1 / math.factorial(k) for k in range(n))
    assert math.isclose(euler_infinite_sum(n), expected, rel_tol=1e-14, abs_tol=1e-14)
    assert isinstance(euler_infinite_sum(n), float)
table = captured_output(print_euler_sum_table, 4).splitlines()
assert len(table) == 4
assert [float(value) for value in table] == [euler_infinite_sum(k) for k in range(1, 5)]
for epsilon in (0.1, 1e-3, 1e-6, 1e-12):
    assert abs(euler_approx(epsilon) - math.exp(1)) < epsilon
for b in range(1, 11):
    # 对 n<=300，b>=2 时只需枚举 0 到 9 次幂；b=1 的幂恒等于1。
    expected_powers = {b ** exponent for exponent in range(10)}
    for n in range(1, 301):
        assert is_power(b, n) == (n in expected_powers), (b, n)
for n in range(1, 301):
    assert is_power_of_2(n) == is_power(2, n)
assert is_power_of_2(2 ** 100) and not is_power_of_2(2 ** 100 + 1)
print("全部检查通过：打印格式、GCD、欧拉数近似及整数幂判断。")

全部检查通过：打印格式、GCD、欧拉数近似及整数幂判断。


## 作业总结
指定三组 GCD 分别为 **1、300、60**。本实现对负数先取绝对值，结果保持非负。
欧拉级数的“前 n 项”从 k=0 开始，因此 n=0 返回 0.0，n=4 返回约 2.666667。
整数幂判断依靠重复整除，1 对应零次幂，底数为 1 时单独处理。